# Gov24 캡차 이미지 수집

`https://www.gov.kr/mw/captcha` API를 호출하여 캡차 이미지를 다운로드합니다.

In [5]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO

In [6]:
# 저장 경로 설정
base_path = Path("captcha_data/gov24/1/images/draft")
base_path.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {base_path.absolute()}")

저장 경로: c:\work\hyper-captcha-resolver\captcha_data\gov24\1\images\draft


In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

captcha_id = 'gov24'
rev = 1
backend = 'keras'

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
train_data.rev = rev
image_width: int = 200
image_height: int = 50
engine.batch_predict_model(model=model)
model_path = train_data.get_model_path()
image_path = train_data.choice_pred_image()
pred, confidence = engine.predict(model=model, image_path=image_path)
print("image_path : ", image_path)
print("pred : ", pred)
print("confidence : ", f'{confidence:.4f}')
print("Done!")


In [7]:
def download_captcha(save_path: Path, index: int, total: int) -> bool:
    """
    캡차 이미지를 다운로드하고 저장합니다.
    
    Args:
        save_path: 저장할 경로
        index: 현재 인덱스
        total: 전체 개수
    
    Returns:
        성공 여부
    """
    url = "https://www.gov.kr/mw/captcha"
    
    try:
        # API 호출
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # 타임스탬프 기반 파일명 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = save_path / filename
        
        # 이미지로 변환 후 PNG로 저장
        image = Image.open(BytesIO(response.content))
        image.save(filepath, format="PNG")
        
        print(f"[{index + 1}/{total}] 저장 완료: {filename}")
        return True
        
    except Exception as e:
        print(f"[{index + 1}/{total}] 오류 발생: {e}")
        return False

In [8]:
# 다운로드 설정
TARGET_COUNT = 50  # 다운로드할 이미지 개수
DELAY = 0.25         # 요청 간 대기 시간 (초)

print(f"캡차 이미지 {TARGET_COUNT}개 다운로드 시작...")
print(f"요청 간격: {DELAY}초")
print("-" * 50)

캡차 이미지 50개 다운로드 시작...
요청 간격: 0.25초
--------------------------------------------------


In [10]:
# 이미지 다운로드 실행
success_count = 0
fail_count = 0

for i in range(TARGET_COUNT):
    if download_captcha(base_path, i, TARGET_COUNT):
        success_count += 1
    else:
        fail_count += 1
    
    # 마지막 요청이 아니면 대기
    if i < TARGET_COUNT - 1:
        time.sleep(DELAY)

print("-" * 50)
print(f"\n다운로드 완료!")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 위치: {base_path.absolute()}")

[1/50] 저장 완료: 20251106_103550_419653.png
[2/50] 저장 완료: 20251106_103551_160480.png
[3/50] 저장 완료: 20251106_103551_898302.png
[4/50] 저장 완료: 20251106_103552_654609.png
[5/50] 저장 완료: 20251106_103553_364621.png
[6/50] 저장 완료: 20251106_103554_120949.png
[7/50] 저장 완료: 20251106_103554_861158.png
[8/50] 저장 완료: 20251106_103555_811465.png
[9/50] 저장 완료: 20251106_103556_729779.png
[10/50] 저장 완료: 20251106_103557_739133.png
[11/50] 저장 완료: 20251106_103601_741902.png
[12/50] 저장 완료: 20251106_103602_690503.png
[13/50] 저장 완료: 20251106_103603_608185.png
[14/50] 저장 완료: 20251106_103604_555208.png
[15/50] 저장 완료: 20251106_103605_569543.png
[16/50] 저장 완료: 20251106_103606_522366.png
[17/50] 저장 완료: 20251106_103607_595155.png
[18/50] 저장 완료: 20251106_103608_639269.png
[19/50] 저장 완료: 20251106_103609_631943.png
[20/50] 저장 완료: 20251106_103610_597184.png
[21/50] 저장 완료: 20251106_103611_617459.png
[22/50] 저장 완료: 20251106_103612_586824.png
[23/50] 저장 완료: 20251106_103613_608214.png
[24/50] 저장 완료: 20251106_103614_664820.png
[

### 캡차 이미지 인식 및 파일명 변경

학습된 모델을 사용하여 draft 폴더의 이미지를 인식하고, 예측된 레이블로 파일명을 변경합니다.

In [11]:
import os
from pathlib import Path
import tensorflow as tf
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

draft_image_dir = "captcha_data/gov24/1/images/draft"
draft_image_files = sorted([str(p) for p in Path(draft_image_dir).glob("*.png")])
print(f"Draft 이미지 개수: {len(draft_image_files)}")

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height
keras_model: KerasModel = model
matched = 0
pred_img_path_list = keras_model.train_data.get_data_files(train=False)
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

for idx, img_path in enumerate(draft_image_files):
    pred, confidence = engine.predict(model=model, image_path=img_path, verbose=0)
    # rename draft image with predicted text
    new_image_path = os.path.join(draft_image_dir, pred + ".png")
    if(os.path.exists(new_image_path)):
        continue
    Path(img_path).rename(new_image_path)
    print(f"[{idx + 1}/{len(draft_image_files)}] image_path : {img_path}")
    print(f"pred : {pred}")
    print(f"confidence : {confidence:.4f}")
    print("new_image_path : ", new_image_path)
    print("Done!")


Draft 이미지 개수: 100

[1/100] image_path : captcha_data\gov24\1\images\draft\20251106_103457_100731.png
pred : 802127
confidence : 0.9923
new_image_path :  captcha_data/gov24/1/images/draft\802127.png
Done!
[2/100] image_path : captcha_data\gov24\1\images\draft\20251106_103457_866421.png
pred : 889614
confidence : 0.9937
new_image_path :  captcha_data/gov24/1/images/draft\889614.png
Done!
[3/100] image_path : captcha_data\gov24\1\images\draft\20251106_103458_641097.png
pred : 276061
confidence : 0.9908
new_image_path :  captcha_data/gov24/1/images/draft\276061.png
Done!
[4/100] image_path : captcha_data\gov24\1\images\draft\20251106_103459_388159.png
pred : 667014
confidence : 0.9884
new_image_path :  captcha_data/gov24/1/images/draft\667014.png
Done!
[5/100] image_path : captcha_data\gov24\1\images\draft\20251106_103500_165463.png
pred : 797472
confidence : 0.9741
new_image_path :  captcha_data/gov24/1/images/draft\797472.png
Done!
[6/100] image_path : captcha_data\gov24\1\images\draft\2

In [ ]:
# 이미지 리사이즈 및 크롭 (오른쪽 여백 제거)
from pathlib import Path
from PIL import Image

# 경로 설정
source_dir = Path("captcha_data/gov24/0/images/draft-thin")
target_dir = Path("captcha_data/gov24/0/images/resize")
target_dir.mkdir(parents=True, exist_ok=True)

# 이미지 파일 목록
image_files = list(source_dir.glob("*.png"))
print(f"처리할 이미지 개수: {len(image_files)}")

# 각 이미지 처리
success_count = 0
for idx, img_path in enumerate(image_files, 1):
    try:
        # 이미지 열기 (221 * 81)
        img = Image.open(img_path)

        # 위쪽, 왼쪽 1픽셀 크롭
        img_cropped = img.crop((1, 1, 221, 81))
        
        # 225x50으로 리사이즈
        img_resized = img_cropped.resize((200, 50), Image.Resampling.LANCZOS)
        
        # 저장 (파일명 유지)
        target_path = target_dir / img_path.name
        img_resized.save(target_path, format="PNG")
        
        success_count += 1
        
        if idx % 50 == 0 or idx == len(image_files):
            print(f"[{idx}/{len(image_files)}] 처리 완료")
            
    except Exception as e:
        print(f"[{idx}/{len(image_files)}] 오류 발생 ({img_path.name}): {e}")

print(f"\n작업 완료!")
print(f"성공: {success_count}/{len(image_files)}")
print(f"저장 위치: {target_dir.absolute()}")

In [ ]:
# target_dir = Path("captcha_data/gov24/0/images/resize")
import glob


image_list = glob.glob(os.path.join(target_dir, "*.png"))
print(f"리사이즈된 이미지 개수: {len(image_list)}")

In [ ]:
import os, glob, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import keras
import tensorflow as tf

from pathlib import Path
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

draft_dir = Path("captcha_data/gov24/1/images/draft")
target_dir = Path("captcha_data/gov24/1/images/labeled")
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

start = time.time()

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height

keras_model: KerasModel = model
matched = 0

pred_img_path_list = sorted(glob.glob(os.path.join(target_dir, "*.png")))
pred_labels = [os.path.basename(p).split(".")[0] for p in pred_img_path_list]
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list, pred_labels))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

# Batch prediction
all_preds = []
all_labels = []

for batch in pred_dataset:
    images = batch["image"]
    labels = batch["label"]
    
    # Predict batch
    pred_vals = keras_model.predict_model.predict(images, verbose=0)
    preds = keras_model.decode_batch_predictions(pred_vals)
    
    # Decode original labels
    for label in labels:
        label_text = tf.strings.reduce_join(
            keras_model.num_to_char(label + 1)
        ).numpy().decode("utf-8")
        all_labels.append(label_text)
    
    all_preds.extend(preds)

# Compare predictions with original labels
for idx, (ori, pred) in enumerate(zip(all_labels, all_preds)):
    img_path = pred_img_path_list[idx]
    msg = ""
    if ori == pred:
        train_img_path = train_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, train_img_path, overwrite=True)
        matched += 1
    else:
        # 불일치 파일은 pred_dir로 복사
        pred_img_path = pred_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, pred_img_path, overwrite=True)
        msg = " Not matched!"
    
    # Calculate confidence for display (optional)
    print(f"ori: {ori}, pred: {pred}{msg}")

end = time.time()
total = len(pred_img_path_list)
accuracy = matched / total * 100 if total > 0 else 0

print(f"Matched: {matched}, Total: {total}, Accuracy: {accuracy:.2f}%")
print(f"pred time: {end - start:.2f} sec") 


# engine.batch_predict_model(model=model)
# model_path = train_data.get_model_path()
# image_path = train_data.choice_pred_image()
# pred, confidence = engine.predict(model=model, image_path=image_path)
# print("image_path : ", image_path)
# print("pred : ", pred)
# print("confidence : ", f'{confidence:.4f}')
# print("Done!")


In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 이미지 크기 검사
from pathlib import Path
from PIL import Image

# 경로 설정
pred_dir = Path("captcha_data/gov24/1/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 크기가 다른 이미지 리스트
    wrong_size_images = []
    target_width = 200
    target_height = 50
    
    for img_path in image_files:
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            if width != target_width or height != target_height:
                wrong_size_images.append({
                    'name': img_path.name,
                    'size': f"{width}x{height}"
                })
                
        except Exception as e:
            print(f"⚠️  오류 ({img_path.name}): {e}")
    
    # 결과 출력
    if wrong_size_images:
        print(f"\n❌ 크기가 {target_width}x{target_height}이 아닌 이미지 ({len(wrong_size_images)}개):\n")
        for idx, img_info in enumerate(wrong_size_images, 1):
            print(f"  {idx:3d}. {img_info['name']:30s} -> {img_info['size']}")
    else:
        print(f"\n✅ 모든 이미지가 {target_width}x{target_height} 크기입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 ({target_width}x{target_height}): {len(image_files) - len(wrong_size_images)}개")
    print(f"비정상: {len(wrong_size_images)}개")

In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 파일명 길이 검사 (확장자 포함 10자리가 아닌 것)
from pathlib import Path

# 경로 설정
pred_dir = Path("captcha_data/gov24/0/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 파일명 길이가 10자리(확장자 포함)가 아닌 이미지 리스트
    wrong_name_images = []
    target_length = 10  # 예: "abc12.png" = 9자 (레이블 5자 + ".png" 4자)
    
    for img_path in image_files:
        filename = img_path.name
        name_length = len(filename)
        
        if name_length != target_length:
            wrong_name_images.append({
                'name': filename,
                'length': name_length,
                'label_length': len(img_path.stem)  # 확장자 제외한 레이블 길이
            })
    
    # 결과 출력
    if wrong_name_images:
        print(f"\n❌ 파일명 길이가 {target_length}자가 아닌 이미지 ({len(wrong_name_images)}개):\n")
        for idx, img_info in enumerate(wrong_name_images, 1):
            print(f"  {idx:3d}. {img_info['name']:40s} (길이: {img_info['length']:2d}, 레이블: {img_info['label_length']:2d}자)")
    else:
        print(f"\n✅ 모든 이미지 파일명이 {target_length}자입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 (파일명 {target_length}자): {len(image_files) - len(wrong_name_images)}개")
    print(f"비정상: {len(wrong_name_images)}개")

In [ ]:
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32
model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)


#### 학습 데이타 섞기

In [4]:
import captchaResolver.engine as engine

engine.redistribute_train_pred(
    image_dir="captcha_data/gov24/1/images",
    train_ratio=0.9,
    verbose=True
)

[redistribute] Step 0: Normalizing file extensions to '.png'
[redistribute] Step 0: Normalized 0 files
[redistribute] Step 1: Found 284 files in pred, 2169 files in train
[redistribute] Step 2: Moved 284 files from pred to train
                      Overwritten 0 duplicate files
[redistribute] Step 3: Total 2453 files in train after merge
[redistribute] Step 4: Shuffled and split -> keep 2208, move 245 to pred
[redistribute] Step 5-6: Moved 245 files to pred
[redistribute] ===== Summary =====
                 Pred found: 284
                 Train found: 2169
                 Normalized: 0
                 Pred moved: 284
                 Overwritten: 0
                 Total after merge: 2453
                 Final train: 2208
                 Final pred: 245
[redistribute] ==================


{'pred_found': 284,
 'train_found': 2169,
 'normalized': 0,
 'pred_moved': 284,
 'overwritten': 0,
 'total_after_merge': 2453,
 'final_train': 2208,
 'final_pred': 245}